# 16 - Tool Calling with LangGraph

LangGraph turns the tool loop into an explicit **stateful graph**.

You will learn:
- `StateGraph`
- `MessagesState`
- model node
- `ToolNode`
- conditional routing
- loops
- terminal conditions
- why graph structure is useful for agents

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "")
if CHAT_MODEL:
    print("Configured model:", CHAT_MODEL)
else:
    print("Set OPENAI_CHAT_MODEL for live model examples.")

# %pip install -qU langgraph langchain langchain-openai python-dotenv

## 1. Agent loop as a graph

```text
START
  ↓
MODEL
  ↓
tool needed?
 ├── no → END
 └── yes
       ↓
     TOOLS
       ↓
     MODEL
       ↺
```

## 2. Define a tool

Keep tools small and explicit. The graph controls when they run.

In [ ]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    'Evaluate a basic arithmetic expression for educational use.'
    allowed = set('0123456789+-*/(). %')
    if not set(expression) <= allowed:
        return 'Unsupported expression'
    return str(eval(expression, {'__builtins__': {}}, {}))

tools = [calculator]

## 3. Build the graph

`ToolNode` executes tool calls, while `tools_condition` routes back to the model when a tool call exists.

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

if CHAT_MODEL:
    model = ChatOpenAI(model=CHAT_MODEL, temperature=0)
    model_with_tools = model.bind_tools(tools)

    def call_model(state: MessagesState):
        return {'messages': [model_with_tools.invoke(state['messages'])]}

    builder = StateGraph(MessagesState)
    builder.add_node('model', call_model)
    builder.add_node('tools', ToolNode(tools))

    builder.add_edge(START, 'model')
    builder.add_conditional_edges('model', tools_condition)
    builder.add_edge('tools', 'model')

    graph = builder.compile()
    print('Graph compiled.')
else:
    print('Configure OPENAI_CHAT_MODEL first.')

## 4. Invoke

A normal invocation supplies initial state, usually a message list.

In [ ]:
if CHAT_MODEL:
    result = graph.invoke({
        'messages': [
            {'role': 'user', 'content': 'Calculate (25 * 18) + 4.'}
        ]
    })
    print(result['messages'][-1].content)

## 5. Add a step budget

Every agent loop should have a stopping policy.

Why:
- prevents infinite loops
- controls cost
- protects tools
- creates predictable failure modes

In [ ]:
MAX_STEPS = 5

def check_step(step):
    if step >= MAX_STEPS:
        raise RuntimeError('Maximum agent steps reached.')

for step in range(3):
    check_step(step)

print('Budget check passed.')

## 6. Practice

1. Add a `word_count` tool.
2. Add a verifier node.
3. Add a branch that ends when the answer is ready.
4. Add persistence in the next notebook.

## Evaluation

Evaluate:
- correct routing
- tool execution
- repeated tool loop
- terminal condition
- failure handling
- step-budget behavior

Improvement:
- typed state
- retries
- persistent checkpoints
- human interrupts
- dedicated verifier nodes.

## References / current documentation


- LangChain reference: https://reference.langchain.com/python/langchain/overview
- LangGraph reference: https://reference.langchain.com/python/langgraph/overview
- LangGraph persistence: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/persistence.mdx
- LangGraph interrupts: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/interrupts.mdx
- LangChain human-in-the-loop: https://github.com/langchain-ai/docs/blob/main/src/oss/langchain/human-in-the-loop.mdx
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- MCP Python SDK v2: https://py.sdk.modelcontextprotocol.io/
- MCP Python SDK getting started: https://py.sdk.modelcontextprotocol.io/get-started/